# Day 1 - Deliverable 3: Train / Validation / Test Strategy
**Objective:** Define the strict data isolation protocols required to evaluate model generalization and prevent data leakage.

### 1. The Global ML Workflow Context
This strategy directly fulfills two critical steps in our pipeline:
* **Step 6:** Decide the train/validation/test strategy.
* **Step 13:** Freeze the final test set until final evaluation.

Machine learning models are exceptional at memorization. If we evaluate a model on the exact same data it used to learn its parameters, the performance metrics will be artificially inflated. We split our data to simulate the unseen data the model will encounter during production inference.

### 2. The Three-Vault Architecture

#### Vault 1: The Training Set (The Lab)
* **Purpose:** To learn the model parameters ($\theta$).
* **Mechanism:** The optimization engine (like our gradient descent loop) iterates over this data, calculates the loss, and computes the gradient to update weights. 
* **Analogy:** This is the textbook the model uses to study for the exam.

#### Vault 2: The Validation Set (The Practice Exam)
* **Purpose:** To evaluate architectural decisions and tune hyperparameters (like the learning rate $\alpha$ or the number of training iterations). 
* **Mechanism:** We train candidate models on the Training Set, then predict on the Validation Set. If a model is memorizing network traffic instead of generalizing the underlying patterns of Cross-VM Network Attacks, the validation set is where we catch that failure. We compare models and adjust configurations based *only* on validation metrics.
* **Analogy:** This is the practice test. If the student fails, the teacher can adjust the curriculum (hyperparameters) and let them study again.

#### Vault 3: The Test Set (The Final Audit)
* **Purpose:** To provide a strictly unbiased estimate of final production performance.
* **Mechanism:** This data is locked away (Step 13). It does not influence parameter updates, nor does it influence hyperparameter tuning. It is only unlocked at the very end of the project. If the model fails here, you cannot simply tweak a hyperparameter and try again—doing so turns the test set into a second validation set. 
* **Analogy:** This is the final state board exam. You only get to take it once, and the score you get is your official capability.

### 3. Implementation Rules
1. **Random vs. Temporal Splitting:** For standard tabular data, a random split (e.g., 70% Train / 15% Val / 15% Test) is sufficient. For time-series data, the split must be temporal (train on the past, validate on the present, test on the future) to prevent looking ahead in time.
2. **Preprocessing Isolation:** Any feature engineering or scaling (like Min-Max Scaling or Standardization) must be fit *only* on the Training Set. The exact scaling parameters ($\mu$ and $\sigma$) calculated from the Training Set are then applied to the Validation and Test sets.

### 4. Code Implementation: The Strict Split and Preprocessing Isolation
We will split a dataset into 70% Train, 15% Validation, and 15% Test. 
Crucially, we will apply the Z-Score Standardization formula: $z=\frac{x-\mu}{\sigma}$[cite: 1]. We must calculate $\mu$ (mean) and $\sigma$ (standard deviation) **only** from the Training set to prevent data leakage, and then apply those exact parameters to transform the Validation and Test sets.

In [2]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Simulate a dataset with 1000 observations and 5 features
X_full = np.random.rand(1000, 5) * 100 # Features with varying scales
y_full = np.random.randint(0, 2, 1000) # Binary classification target

# 2. First Split: Carve out the Training Set (70%) and a temporary holdout (30%)
X_train, X_temp, y_train, y_temp = train_test_split(
    X_full, y_full, 
    test_size=0.30, 
    random_state=42 # Random state ensures reproducibility 
)

# 3. Second Split: Divide the holdout equally into Validation (15%) and Test (15%)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, 
    test_size=0.50, 
    random_state=42
)

print(f"Training Set:   {X_train.shape[0]} observations (Lab)")
print(f"Validation Set: {X_val.shape[0]} observations (Practice Exam)")
print(f"Test Set:       {X_test.shape[0]} observations (Locked Final Audit)")

# 4. Preprocessing Isolation (Preventing Data Leakage)
scaler = StandardScaler()

# FIT ONLY ON TRAINING DATA to learn its specific mean and variance
scaler.fit(X_train) 

# TRANSFORM all datasets using the parameters learned strictly from the training data
X_train_scaled = scaler.transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Verify the scaling on the training set (Mean should be ~0, Variance should be ~1)
print(f"\nTraining Set Mean after scaling: {np.mean(X_train_scaled):.2f}")
print(f"Training Set Variance after scaling: {np.var(X_train_scaled):.2f}")

Training Set:   700 observations (Lab)
Validation Set: 150 observations (Practice Exam)
Test Set:       150 observations (Locked Final Audit)

Training Set Mean after scaling: -0.00
Training Set Variance after scaling: 1.00
